Język: [English](lab.en.ipynb) | **Polski**

# Laboratorium 09 — klasyfikacja SMS (spam/ham) z RNN/LSTM/GRU

Materiał towarzyszący [`README.pl.md`](README.pl.md). Inny problem niż wykład: prawdziwa
klasyfikacja tekstu (SMS Spam Collection), nie syntetyczne sekwencje.


In [ ]:
import re  # noqa: F401 - uzywane w TODO ponizej
import time
from collections import Counter  # noqa: F401 - uzywane w TODO ponizej
from pathlib import Path

import matplotlib.pyplot as plt  # noqa: F401 - uzywane w TODO ponizej
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence  # noqa: F401 - uzywane w TODO ponizej

torch.manual_seed(0)
torch.set_num_threads(4)
device = torch.device("cpu")
print(f"Uzywane urzadzenie: {device}")


## 1. Dane: SMS Spam Collection


In [ ]:
import hashlib
import os
import urllib.request
import zipfile

_REPO_ROOT = next(
    (
        parent
        for parent in (Path.cwd(), *Path.cwd().parents)
        if (parent / "pyproject.toml").is_file()
    ),
    Path.cwd(),
)
DATA_CACHE_DIR = Path(
    os.environ.get("DEEP_LEARNING_DATA_CACHE", _REPO_ROOT / "data_cache")
).expanduser().resolve()

ZIP_URL = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
EXPECTED_SHA256 = "1587ea43e58e82b14ff1f5425c88e17f8496bfcdb67a583dbff9eefaf9963ce3"
CACHE_DIR = DATA_CACHE_DIR
ZIP_PATH = os.path.join(CACHE_DIR, "sms_spam_collection.zip")
TXT_PATH = os.path.join(CACHE_DIR, "SMSSpamCollection")

os.makedirs(CACHE_DIR, exist_ok=True)
if not os.path.exists(TXT_PATH):
    if not os.path.exists(ZIP_PATH):
        urllib.request.urlretrieve(ZIP_URL, ZIP_PATH)
    with open(ZIP_PATH, "rb") as f:
        actual_sha256 = hashlib.sha256(f.read()).hexdigest()
    assert actual_sha256 == EXPECTED_SHA256, (
        f"SHA-256 mismatch: expected {EXPECTED_SHA256}, got {actual_sha256}"
    )
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extract("SMSSpamCollection", CACHE_DIR)

labels_all, texts_all = [], []
with open(TXT_PATH, encoding="utf-8") as f:
    for line in f:
        label, text = line.rstrip("\n").split("\t", 1)
        labels_all.append(label)
        texts_all.append(text)

print(f"liczba wiadomosci: {len(texts_all)}")
print("przyklad (ham):", next(t for t, lb in zip(texts_all, labels_all) if lb == "ham")[:60])
print("przyklad (spam):", next(t for t, lb in zip(texts_all, labels_all) if lb == "spam")[:60])

In [ ]:
# TODO: policz i wypisz liczbe (i procent) wiadomosci w kazdej klasie (ham/spam)
raise NotImplementedError("TODO: rozklad klas")


_(Twoja odpowiedź tutaj.)_

## 2. Tokenizacja


In [ ]:
def tokenize(text):
    # TODO: zwroc liste tokenow - male litery, slowa jako osobne tokeny, cyfry
    # jako osobne tokeny, interpunkcja jako osobne (pojedyncze) tokeny.
    # Podpowiedz: re.findall z wzorcem alternatywy (slowo | cyfry | pojedynczy
    # znak nie-alfanumeryczny i nie-spacja)
    raise NotImplementedError("TODO: tokenize")


print(tokenize("Free entry! Win a prize NOW, call 0800-123456"))


**Zadanie:** jakie są ograniczenia tak prostej tokenizacji (odmiana słów,
interpunkcja, słowa spoza słownika)?


_(Twoja odpowiedź tutaj.)_

## 3. Podział train/val/test (gotowe)


In [ ]:
g = torch.Generator().manual_seed(0)
perm = torch.randperm(len(texts_all), generator=g).tolist()
n = len(texts_all)
n_train, n_val = int(0.7 * n), int(0.15 * n)
train_idx = perm[:n_train]
val_idx = perm[n_train:n_train + n_val]
test_idx = perm[n_train + n_val:]

train_texts_raw = [texts_all[i] for i in train_idx]
train_labels = [1 if labels_all[i] == "spam" else 0 for i in train_idx]
val_texts_raw = [texts_all[i] for i in val_idx]
val_labels = [1 if labels_all[i] == "spam" else 0 for i in val_idx]
test_texts_raw = [texts_all[i] for i in test_idx]
test_labels = [1 if labels_all[i] == "spam" else 0 for i in test_idx]

train_texts = [tokenize(t) for t in train_texts_raw]
val_texts = [tokenize(t) for t in val_texts_raw]
test_texts = [tokenize(t) for t in test_texts_raw]

print(f"train={len(train_texts)}  val={len(val_texts)}  test={len(test_texts)}")
print(f"train spam ratio={sum(train_labels)/len(train_labels):.3f}")


## 4. Słownik: tylko ze zbioru treningowego


In [ ]:
MIN_FREQ = 2
# TODO:
# 1. policz czestosc kazdego tokenu WYLACZNIE w train_texts (collections.Counter)
# 2. zbuduj vocab: lista ["<PAD>", "<UNK>"] + tokeny o czestosci >= MIN_FREQ
# 3. word_to_id: slownik token -> indeks (pozycja w vocab)
raise NotImplementedError("TODO: budowa slownika")
PAD_ID, UNK_ID = 0, 1


def encode(tokens, max_len=50):
    # TODO: zamien liste tokenow na liste id (word_to_id), token spoza slownika
    # -> UNK_ID; obetnij do max_len
    raise NotImplementedError("TODO: encode")


**Zadanie (kod):** zweryfikuj, że token spoza słownika treningowego nie powoduje
błędu, tylko zamienia się na `<UNK>`.


In [ ]:
fake_token = "zzzznieistniejacyslowo"
# TODO: sprawdz, ze fake_token NIE jest w word_to_id, a encode([fake_token]) nie
# rzuca wyjatku i zwraca UNK_ID
raise NotImplementedError("TODO: weryfikacja UNK")


## 5. Batche o zmiennej długości (gotowe)


In [ ]:
def make_batch_tensors(encoded_texts, labels_subset):
    lengths = torch.tensor([max(len(e), 1) for e in encoded_texts])
    max_len = lengths.max().item()
    padded = torch.full((len(encoded_texts), max_len), PAD_ID, dtype=torch.long)
    for i, e in enumerate(encoded_texts):
        if len(e) > 0:
            padded[i, :len(e)] = torch.tensor(e, dtype=torch.long)
    return padded, lengths, torch.tensor(labels_subset)


train_encoded = [encode(t) for t in train_texts]
val_encoded = [encode(t) for t in val_texts]
test_encoded = [encode(t) for t in test_texts]

demo_X, demo_lengths, demo_y = make_batch_tensors(train_encoded[:2], train_labels[:2])
print(f"dwie wiadomosci roznej dlugosci w jednym batchu: ksztalt={tuple(demo_X.shape)}")
print(f"rzeczywiste dlugosci: {demo_lengths.tolist()}")
shorter_row = demo_X[0] if demo_lengths[0] < demo_lengths[1] else demo_X[1]
print(f"krotsza wiadomosc (dopelniona PAD_ID={PAD_ID} na koncu):\n{shorter_row}")


### Pętla treningowa (gotowe, używana przez wszystkie modele w tym notebooku)


In [ ]:
def train_eval(model_cls, vocab_size, epochs=5, lr=0.001, batch_size=32, seed=0, **model_kwargs):
    torch.manual_seed(seed)
    model = model_cls(vocab_size, **model_kwargs)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    n = len(train_encoded)

    start = time.time()
    history = {"train_loss": [], "val_loss": []}
    for epoch in range(epochs):
        model.train()
        perm_e = torch.randperm(n)
        total_loss = 0.0
        for i in range(0, n, batch_size):
            idx = perm_e[i:i + batch_size].tolist()
            batch_enc = [train_encoded[j] for j in idx]
            batch_lab = [train_labels[j] for j in idx]
            X, lengths, y = make_batch_tensors(batch_enc, batch_lab)
            opt.zero_grad()
            loss = loss_fn(model(X, lengths), y)
            loss.backward()
            opt.step()
            total_loss += loss.item() * len(idx)
        train_loss = total_loss / n
        history["train_loss"].append(train_loss)

        model.eval()
        with torch.no_grad():
            Xv, lv, yv = make_batch_tensors(val_encoded, val_labels)
            val_loss = loss_fn(model(Xv, lv), yv).item()
        history["val_loss"].append(val_loss)
    train_time = time.time() - start

    model.eval()
    with torch.no_grad():
        Xv, lv, yv = make_batch_tensors(val_encoded, val_labels)
        val_acc = (model(Xv, lv).argmax(1) == yv).float().mean().item()

    return model, history, val_acc, train_time


## 6. Baseline bez kolejności


In [ ]:
D_EMBED = 32


class MeanPoolBaseline(nn.Module):
    def __init__(self, vocab_size, d_embed=D_EMBED, n_classes=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_embed, padding_idx=PAD_ID)
        self.fc = nn.Linear(d_embed, n_classes)

    def forward(self, x, lengths):
        # TODO:
        # 1. e = self.embed(x)  (B, T, d_embed)
        # 2. zbuduj maske (x != PAD_ID), zeby WYKLUCZYC padding z usredniania
        # 3. policz srednia embeddingow TYLKO po prawdziwych tokenach (uzyj
        #    lengths, nie liczby wszystkich kolumn T - inaczej padding
        #    "rozwadnia" wynik krotszych wiadomosci)
        # 4. zwroc self.fc(pooled)
        raise NotImplementedError("TODO: MeanPoolBaseline.forward")


baseline_model, baseline_history, baseline_val_acc, baseline_time = train_eval(
    MeanPoolBaseline, len(vocab)  # noqa: F821 - vocab zdefiniowany w sekcji 4
)
print(f"baseline (mean-pool): val_acc={baseline_val_acc:.3f}  czas={baseline_time:.2f}s  "
      f"params={sum(p.numel() for p in baseline_model.parameters())}")


## 7. Klasyfikator `nn.Embedding` + `nn.GRU`


In [ ]:
D_HIDDEN = 32


class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, d_embed=D_EMBED, d_hidden=D_HIDDEN, n_classes=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_embed, padding_idx=PAD_ID)
        self.gru = nn.GRU(d_embed, d_hidden, batch_first=True)
        self.fc = nn.Linear(d_hidden, n_classes)

    def forward(self, x, lengths):
        # TODO:
        # 1. e = self.embed(x)
        # 2. zapakuj e przez pack_padded_sequence (batch_first=True,
        #    enforce_sorted=False, lengths.cpu()) PRZED podaniem do self.gru -
        #    to kluczowy krok, ktory sprawia, ze GRU NIE przetwarza paddingu
        # 3. self.gru(packed) zwraca (output, h_n) - h_n to stan PO OSTATNIM
        #    PRAWDZIWYM tokenie kazdej sekwencji (nie po ostatniej kolumnie
        #    tensora!)
        # 4. zwroc self.fc(h_n.squeeze(0))
        raise NotImplementedError("TODO: GRUClassifier.forward")


**Weryfikacja: ten sam tekst musi dać tę samą reprezentację niezależnie od tego, z
czym jest zbatchowany.**


In [ ]:
torch.manual_seed(0)
check_model = GRUClassifier(len(vocab))  # noqa: F821 - vocab zdefiniowany w sekcji 4
check_model.eval()

single_text = train_encoded[0]
longer_text = max(train_encoded, key=len)

X_alone, len_alone, _ = make_batch_tensors([single_text], [0])
X_batched, len_batched, _ = make_batch_tensors([single_text, longer_text], [0, 0])

with torch.no_grad():
    pred_alone = check_model(X_alone, len_alone)
    pred_batched = check_model(X_batched, len_batched)[0:1]

match = torch.allclose(pred_alone, pred_batched, atol=1e-5)
print(f"ta sama wiadomosc, ten sam wynik niezaleznie od paddingu: {match}")
assert match, "BLAD: reprezentacja zalezy od paddingu - sprawdz obsluge lengths w forward()"


## 8. Trening klasyfikatora z sekcji 7 (pętla z sekcji 5 — gotowe wywołanie)


In [ ]:
gru_model, gru_history, gru_val_acc, gru_time = train_eval(
    GRUClassifier, len(vocab)  # noqa: F821 - vocab zdefiniowany w sekcji 4
)
print(f"GRU: val_acc={gru_val_acc:.3f}  czas={gru_time:.2f}s")
print(f"strata treningowa (ostatnia epoka): {gru_history['train_loss'][-1]:.4f}")
print(f"strata walidacyjna (ostatnia epoka): {gru_history['val_loss'][-1]:.4f}")


## 9. Porównanie ≥2 konfiguracji


**Zadanie:** jakie dwie konfiguracje porównujesz i dlaczego akurat te?

_(Twoja odpowiedź tutaj.)_

In [ ]:
# TODO:
# 1. napisz LSTMClassifier, analogicznie do GRUClassifier (sekcja 7), ale z
#    nn.LSTM - pamietaj, ze LSTM zwraca (output, (h_n, c_n)), nie (output, h_n)
# 2. wytrenuj go przez train_eval (ten sam wzorzec co dla GRU)
# 3. wypisz obok siebie: val_acc, czas treningu, liczbe parametrow dla OBU
raise NotImplementedError("TODO: druga konfiguracja i porownanie")


## 10. Ewaluacja na zbiorze testowym


In [ ]:
def macro_f1(preds, targets, n_classes=2):
    # TODO: policz F1 dla kazdej klasy osobno (precision, recall, F1 z tp/fp/fn),
    # zwroc SREDNIA (macro-average, nie wazona liczba przykladow)
    raise NotImplementedError("TODO: macro_f1")


# TODO:
# 1. wybierz lepszy model wedlug walidacji (gru_val_acc vs lstm_val_acc)
# 2. policz predykcje NA ZBIORZE TESTOWYM (pierwszy raz w tym notebooku)
# 3. policz test_acc, test_macro_f1 (uzyj macro_f1 powyzej)
# 4. zbuduj i narysuj confusion matrix 2x2
raise NotImplementedError("TODO: ewaluacja testowa")


## 11. Zadanie diagnostyczne

Poniższy kod **wygląda** poprawnie — nie rzuca wyjątku, trenuje się — ale ma błąd
związany z obsługą paddingu.


In [ ]:
class BuggyClassifier(nn.Module):
    def __init__(self, vocab_size, d_embed=D_EMBED, d_hidden=D_HIDDEN, n_classes=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_embed, padding_idx=PAD_ID)
        self.gru = nn.GRU(d_embed, d_hidden, batch_first=True)
        self.fc = nn.Linear(d_hidden, n_classes)

    def forward(self, x, lengths):
        e = self.embed(x)
        out, _ = self.gru(e)  # BRAK pack_padded_sequence!
        last_step = out[:, -1, :]  # zawsze OSTATNIA KOLUMNA tensora
        return self.fc(last_step)


torch.manual_seed(0)
buggy_model = BuggyClassifier(len(vocab))  # noqa: F821 - vocab zdefiniowany w sekcji 4
buggy_model.eval()

X_alone_b, len_alone_b, _ = make_batch_tensors([single_text], [0])
X_batched_b, len_batched_b, _ = make_batch_tensors([single_text, longer_text], [0, 0])

with torch.no_grad():
    pred_alone_b = buggy_model(X_alone_b, len_alone_b)
    pred_batched_b = buggy_model(X_batched_b, len_batched_b)[0:1]

print("ten sam tekst, BuggyClassifier, batchowany osobno vs z dluzsza sekwencja:")
print("  osobno:  ", pred_alone_b.flatten().tolist())
print("  w batchu:", pred_batched_b.flatten().tolist())
print("  identyczne?", torch.allclose(pred_alone_b, pred_batched_b, atol=1e-5))


**Zadanie:** zdiagnozuj błąd (konkretna linia), wyjaśnij, kiedy się ujawnia (sprawdź
dla batch_size=1 osobno), popraw kod, zweryfikuj na sekwencjach różnej długości.


In [ ]:
# TODO:
# 1. napisz diagnoze (jako komentarz): ktora linia jest zrodlem bledu i dlaczego
# 2. napisz FixedClassifier - poprawiona wersja BuggyClassifier (uzyj
#    pack_padded_sequence, jak w sekcji 7)
# 3. zweryfikuj analogicznie do sekcji 7 (ten sam tekst, osobno vs w batchu)
raise NotImplementedError("TODO: diagnoza i poprawka")


## 12. Analiza błędów


In [ ]:
# TODO:
# 1. znajdz indeksy blednie sklasyfikowanych przykladow testowych
# 2. wypisz 5-8 z nich (tekst + prawdziwa/przewidziana etykieta)
# 3. policz, ile bledow to ham->spam a ile spam->ham
# 4. porownaj srednia dlugosc (w tokenach) wiadomosci poprawnie i blednie
#    sklasyfikowanych
raise NotImplementedError("TODO: analiza bledow")


_(Twoja odpowiedź tutaj.)_

## 13. Wnioski


**1. O ile model sekwencyjny przebija baseline? Co to mówi o tym, na ile klasyfikacja spamu wymaga rozumienia kolejności, w porównaniu z zadaniem z wykładu?**

_(Twoja odpowiedź tutaj.)_

**2. Która konfiguracja z sekcji 9 wypadła lepiej? Czy różnica jest przekonująca?**

_(Twoja odpowiedź tutaj.)_

**3. Jakie jest co najmniej jedno ograniczenie tego eksperymentu?**

_(Twoja odpowiedź tutaj.)_